# ML Live Interview — Boilerplate Notebook

**How to use in the interview:**
1. Put the provided data file next to this notebook (or paste its URL).
2. Edit the 4 vars in Cell 2 (`DATA_PATH`, `TARGET_COL`, `DATE_COL`, `ID_COLS`).
3. Run all cells top-to-bottom, narrate what you see.
4. End on the Hypotheses/Mitigations template (last cells) — that is what is graded.

> You may use an AI agent throughout — narrate your reasoning out loud.

In [ ]:
# Cell 1 — imports & display (run once, no edits needed)
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

print("pandas", pd.__version__)
print("ready — now edit Cell 2 and run all")

In [ ]:
# Cell 2 — ★ ONLY CELL YOU MUST EDIT ★
DATA_PATH  = "data.csv"   # local path OR http(s) URL to .csv / .parquet
TARGET_COL = "target"      # <-- set to the label column once you know it (or None to just profile)
DATE_COL   = None           # e.g. "date" / "timestamp" if time-aware, else None
ID_COLS    = []             # e.g. ["id", "user_id"] — excluded from model + flagged as leakage suspects

TEST_SIZE  = 0.2
RANDOM_STATE = 42

In [ ]:
# Cell 3 — universal loader (csv/parquet, local/URL) + first look
def load_data(path: str) -> pd.DataFrame:
    p = str(path)
    is_url = p.startswith(("http://", "https://"))
    ext = p.split("?")[0].lower().rsplit(".", 1)[-1] if "." in p.split("?")[0] else ""
    if ext == "parquet":
        assert not is_url, "parquet-over-URL needs fsspec — download it locally first"
        return pd.read_parquet(p)
    return pd.read_csv(p)  # csv works for local paths AND urls

try:
    df = load_data(DATA_PATH)
except FileNotFoundError:
    # fallback: tiny synthetic demo so the rest of the notebook still runs before data arrives
    print(f"⚠ '{DATA_PATH}' not found — using synthetic demo data so cells still run.")
    rng = np.random.default_rng(42)
    n = 1000
    df = pd.DataFrame({
        "num_a": rng.normal(0, 1, n),
        "num_b": rng.exponential(1.0, n),
        "cat_c": rng.choice(["a", "b", "c", None], n, p=[.4, .3, .2, .1]),
        "date": pd.date_range("2024-01-01", periods=n, freq="D"),
    })
    df["target"] = ((df["num_a"] + rng.normal(0, .5, n)) > 0).astype(int)
    df.loc[rng.choice(n, 50, replace=False), "num_b"] = np.nan
    if TARGET_COL is None:
        TARGET_COL = "target"
    if DATE_COL is None:
        DATE_COL = "date"

print(f"shape: {df.shape}")
display(df.head())
print("\ndtypes:")
print(df.dtypes)

In [ ]:
# Cell 4 — data profile: shape / missing / duplicates / memory / constants
print(f"rows={df.shape[0]:,}, cols={df.shape[1]}, memory={df.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"duplicated rows: {df.duplicated().sum():,} ({df.duplicated().mean():.2%})")

prof = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "pct_missing": df.isna().mean().round(4),
    "n_unique": df.nunique(dropna=False),
})
prof["is_constant"] = prof["n_unique"] <= 1
prof["is_id_like"] = (prof["n_unique"] >= 0.95 * len(df)) & (len(df) > 10)
display(prof.sort_values("pct_missing", ascending=False))

# SAY OUT LOUD: which missingness looks MCAR vs systematic? which cols are IDs/constants?

In [ ]:
# Cell 5 — missingness chart + missing-vs-target signal
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
if len(miss):
    plt.figure(figsize=(10, max(2.5, .4*len(miss))))
    miss.plot(kind="barh")
    plt.xlabel("fraction missing"); plt.title("Missingness by column"); plt.tight_layout(); plt.show()
else:
    print("No missing values.")

# Is missingness itself predictive? (quick MNAR check)
if TARGET_COL in df.columns and len(miss):
    for c in miss.index[:5]:
        g = df.groupby(df[c].isna())[TARGET_COL].mean(numeric_only=True)
        print(f"\nP(target|{c} missing) vs present:")
        print(g)
    print("\n→ If these differ a lot: missingness is INFORMATIVE (keep a missing-indicator, don't just impute).")

In [ ]:
# Cell 6 — target analysis (auto: classification vs regression)
if TARGET_COL not in df.columns:
    print(f"TARGET_COL='{TARGET_COL}' not in data — set it in Cell 2, skipping target analysis.")
    TASK = None
else:
    y = df[TARGET_COL]
    nuniq = y.nunique(dropna=True)
    TASK = "classification" if (pd.api.types.is_bool_dtype(y) or pd.api.types.is_object_dtype(y) or nuniq <= 20) else "regression"
    print(f"detected TASK = {TASK}  (target '{TARGET_COL}', {nuniq} unique values)")
    print(y.describe(include='all').to_string())
    print(f"\nmissing in target: {y.isna().sum()} ({y.isna().mean():.2%})")

    fig, ax = plt.subplots(figsize=(8, 3.5))
    if TASK == "classification":
        vc = y.value_counts(dropna=False, normalize=True).sort_index()
        vc.plot(kind="bar", ax=ax)
        ax.set_title(f"Target distribution — imbalance ratio {vc.max()/vc.min():.1f}:1"); ax.set_ylabel("share")
        print("\nclass balance (share):\n", vc.round(4).to_string())
        if vc.min() < 0.10:
            print("→ MINORITY <10%: say 'imbalanced — stratify splits, report PR/F1 not just accuracy'.")
    else:
        ax.hist(y.dropna(), bins=50)
        ax.set_title("Target distribution (regression)")
        skew = y.skew()
        print(f"skew={skew:.2f}", "→ |skew|>1? mention log-transform." if abs(skew) > 1 else "")
        print(f"zeros: {(y==0).mean():.2%}  → many zeros? mention zero-inflation / MAPE instability.")
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 7 — numeric features: distributions + outliers
num_cols = df.select_dtypes(include="number").columns.drop(TARGET_COL, errors="ignore").tolist()
print(f"{len(num_cols)} numeric cols")
if num_cols:
    print(df[num_cols].describe().T.round(3).to_string())
    k = min(len(num_cols), 6)
    fig, axes = plt.subplots(1, k, figsize=(4*k, 3))
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, num_cols[:k]):
        ax.hist(df[c].dropna(), bins=30); ax.set_title(c)
    plt.suptitle(f"Histograms (first {k} numeric)"); plt.tight_layout(); plt.show()

    # outlier quick-scan via IQR
    out = {}
    for c in num_cols:
        s = df[c].dropna()
        if len(s) == 0: continue
        q1, q3 = s.quantile(.25), s.quantile(.75)
        iqr = q3 - q1
        if iqr > 0:
            out[c] = ((s < q1-1.5*iqr) | (s > q3+1.5*iqr)).mean()
    out = pd.Series(out).sort_values(ascending=False)
    print("\noutlier share (IQR rule):\n", out.round(4).to_string())
    print("→ >1-2% outliers? say 'check data entry vs heavy tail; prefer robust scaler / tree model'.")

In [ ]:
# Cell 8 — categoricals: cardinality + top values
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.drop(TARGET_COL, errors="ignore").tolist()
# low-cardinality numerics are often really categoricals
for c in df.select_dtypes(include='number').columns:
    if c != TARGET_COL and df[c].nunique(dropna=True) <= 10:
        cat_cols.append(c)
print(f"{len(cat_cols)} categorical-ish cols")
for c in cat_cols[:12]:
    vc = df[c].value_counts(dropna=False).head(8)
    print(f"\n{c} — {df[c].nunique(dropna=False)} unique (top 8):")
    print((vc / len(df)).round(4).to_string())
    if df[c].nunique(dropna=False) == len(df):
        print("  ⚠ unique per row → ID-like, drop from features.")
    elif df[c].nunique(dropna=False) > 50:
        print("  ⚠ high cardinality → mention target/frequency encoding or hashing.")
print("\n→ Rare levels (<1%)? say 'group into Other to avoid overfitting'.")

In [ ]:
# Cell 9 — correlations + leakage suspects
if num_cols and TARGET_COL in df.columns:
    tmp = df[num_cols + [TARGET_COL]].copy()
    if TASK == "classification":
        tmp[TARGET_COL] = pd.factorize(tmp[TARGET_COL])[0]
    corr = tmp.corr(numeric_only=True)[TARGET_COL].drop(TARGET_COL, errors="ignore").sort_values(key=abs, ascending=False)
    print("top |corr| with target:\n", corr.head(10).round(3).to_string())
    sus = corr[corr.abs() > 0.9]
    if len(sus):
        print(f"\n⚠ LEAKAGE SUSPECTS (|r|>0.9): {list(sus.index)} — say 'computed after outcome? check timestamps'.")
    plt.figure(figsize=(8, max(3, .35*min(len(num_cols), 15))))
    sns.heatmap(tmp[num_cols[:15]].corr(), cmap="coolwarm", center=0, square=False)
    plt.title("Feature correlation (multicollinearity check)"); plt.tight_layout(); plt.show()
    print("→ Pairs with |r|>0.8? say 'drop one / regularize / PCA'.")

# structural leakage checks (work even without target)
print("\nconstant cols:", [c for c in df.columns if df[c].nunique(dropna=False) <= 1])
print("ID-like cols:", [c for c in df.columns if df[c].nunique(dropna=False) == len(df)] + list(ID_COLS))
print("→ Ask: 'which columns would NOT be known at prediction time?' — that is your leakage list.")

In [ ]:
# Cell 10 — time / drift check (only if DATE_COL set)
if DATE_COL and DATE_COL in df.columns:
    d = df.copy()
    d[DATE_COL] = pd.to_datetime(d[DATE_COL], errors="coerce")
    print(f"date range: {d[DATE_COL].min()} → {d[DATE_COL].max()}, NaT={d[DATE_COL].isna().sum()}")
    d = d.sort_values(DATE_COL)
    d["__period"] = pd.qcut(d[DATE_COL].rank(method="first"), 4, labels=["Q1","Q2","Q3","Q4"], duplicates="drop")
    if TARGET_COL in d.columns:
        print("\ntarget mean by time quartile:")
        print(d.groupby("__period", observed=True)[TARGET_COL].mean(numeric_only=True).round(4).to_string())
        fig, ax = plt.subplots()
        d.groupby("__period", observed=True)[TARGET_COL].mean(numeric_only=True).plot(kind="bar", ax=ax)
        ax.set_title("Target drift over time"); plt.tight_layout(); plt.show()
        print("→ Trend/shift across quartiles? say 'non-stationary — split by TIME not randomly, monitor PSI'.")
    for c in num_cols[:4]:
        print(f"\n{c} mean by quartile:\n", d.groupby("__period", observed=True)[c].mean().round(3).to_string())
else:
    print("DATE_COL not set — skipping time/drift check.")
    print("→ Still say: 'if this data has a time axis, random splits overstate performance — use time split.'")

In [ ]:
# Cell 11 — 5-minute baseline (proves signal; gives numbers to reason about)
if TARGET_COL not in df.columns:
    print("Set TARGET_COL first.")
else:
    feat = [c for c in df.columns if c not in [TARGET_COL, DATE_COL] + list(ID_COLS)]
    X, y = df[feat], df[TARGET_COL]
    num_f = [c for c in feat if pd.api.types.is_numeric_dtype(df[c])]
    cat_f = [c for c in feat if c not in num_f]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_f),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                            ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01))]), cat_f),
    ])
    strat = y if (TASK == "classification" and y.nunique() > 1) else None
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=strat)
    model = (RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
             if TASK == "classification" else RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
    pipe = Pipeline([("pre", pre), ("model", model)])
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    print(f"baseline: RF on {len(feat)} features, train={len(Xtr):,}, test={len(Xte):,}")
    if TASK == "classification":
        print(f"accuracy={accuracy_score(yte, pred):.3f}  precision={precision_score(yte, pred, average='weighted', zero_division=0):.3f}  "
              f"recall={recall_score(yte, pred, average='weighted', zero_division=0):.3f}  f1={f1_score(yte, pred, average='weighted', zero_division=0):.3f}")
        try:
            proba = pipe.predict_proba(Xte)
            print(f"roc_auc={(roc_auc_score(yte, proba, multi_class='ovr') if proba.shape[1] > 2 else roc_auc_score(yte, proba[:, 1])):.3f}")
        except Exception as e:
            print("(roc_auc skipped:", e, ")")
        print("confusion_matrix:\n", confusion_matrix(yte, pred))
    else:
        rmse = mean_squared_error(yte, pred) ** .5
        print(f"MAE={mean_absolute_error(yte, pred):.3f}  RMSE={rmse:.3f}  R2={r2_score(yte, pred):.3f}")
    # importances → what to talk about
    try:
        names = pipe.named_steps["pre"].get_feature_names_out()
        imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False)
        print("\ntop 10 importances:\n", imp.head(10).round(4).to_string())
        imp.head(10).sort_values().plot(kind="barh"); plt.title("Top 10 feature importances"); plt.tight_layout(); plt.show()
    except Exception as e:
        print("(importances skipped:", e, ")")
    print("\n→ SAY: 'this is a floor, not a claim. Gap between train/test = variance; low both = bias/data problem.'")

## Diagnosis template — read this out loud, fill the table

For each symptom you found above, name **one likely cause → one check → one mitigation** (no implementation needed).

| # | Symptom (what you saw) | Likely cause | Check (1 line) | Mitigation |
|---|------------------------|--------------|----------------|------------|
| 1 | e.g. target drift Q1→Q4 | seasonality / policy change | plot target by week + join deploy log | time-based split, retrain cadence, monitor PSI |
| 2 | e.g. col X 30% missing, missing ⇒ higher target | MNAR / opt-in field | missing-indicator lift | keep indicator + impute, don't drop rows |
| 3 | e.g. feature r=0.97 with target | leakage (post-outcome) | check feature timestamp vs label time | drop / time-travel-safe rebuild |
| 4 | | | | |
| 5 | | | | |

**Mitigation menu (pick, don't recite):** missing indicators + imputation · group rares · drop IDs/constants ·
time-split + walk-forward · stratified CV · class weights / threshold tuning · robust scaling ·
target/frequency encoding for high-cardinality · calibration · PSI/drift monitors · fairness slice checks.

**Closing line:** “Top 3 risks are ___, ___, ___; I'd first verify ___ because it decides between ___ and ___.”